# 第19講：特徴量の概念

機械学習（AI）の予測精度は、アルゴリズムの複雑さ以上に**「どんな特徴量（手がかり）をAIに入力するか」**で決まる。
このノートブックでは、高校の擬似データ（`mhigh.csv`・677人）や画像・音の数値データを使いながら、次の4つを順番に体験する。

1. **データ操作と特徴量の作成**：`loc`・`iloc`・条件抽出（`&`, `|`）と、新しい特徴量（`bmi`, `class_num`）の作成
2. **構造化データの特徴量選択**：どの特徴量をAIに入れるかで、テストデータの予測誤差（**MAE**）がどう劇的に変わるか
3. **画像データと輪郭フィルター（畳み込みの基礎）**：画像を「数字の行列」として作り、隣り合うピクセルの差から縦・横の輪郭（エッジ）を抽出する
4. **音声データとフーリエ変換**：2つのサイン波が混ざったギザギザな波から、フーリエ変換で元の周波数（50Hzと80Hz）を取り出す


## 0. 準備

最初に1回だけ実行する。グラフの軸に日本語を出すため、Noto Sans JP を登録する。


In [ ]:
import piplite
import pyodide
await piplite.install("seaborn")

import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
%matplotlib inline

from pyodide.http import pyfetch
resp = await pyfetch(
    "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/NotoSansJP%5Bwght%5D.ttf"
)
with open("NotoSansJP.ttf", "wb") as f:
    f.write(await resp.bytes())
fm.fontManager.addfont("NotoSansJP.ttf")

sns.set_theme(style="darkgrid", font="Noto Sans JP")


## 1. データ操作と新しい特徴量の作成

`mhigh.csv`（生徒677人）を読み込み、列名を英語に揃える。


In [ ]:
url = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/mhigh.csv"
data = pd.read_csv(pyodide.http.open_url(url))
data.columns = ["name", "katakana", "sex", "grade",
                "japanese", "math", "total", "height", "weight", "class"]
data.head()


### インデックス（`loc` と `iloc`）と条件抽出の確認

- `loc[行ラベル, 列名]`：列を**名前**で安全に指定する（スライスの終わりを含む）
- `iloc[行番号, 列番号]`：0始まりの**位置番号**で指定する（スライスの終わりを含まない）
- 複数条件の絞り込み：各条件を `( )` で囲み、「かつ」は `&`、「または」は `|` でつなぐ


In [ ]:
# loc で 0〜2行目の name, japanese, math, class 列を取り出す
print(data.loc[0:2, ["name", "japanese", "math", "class"]])

# 条件抽出：3年生 かつ 数学が80点以上の生徒の人数
high_math_3 = data[(data["grade"] == 3) & (data["math"] >= 80)]
print("3年生かつ数学80点以上の人数:", len(high_math_3), "人")


### 生データから新しい特徴量（`bmi` と `class_num`）を作る

- 身長（`height`）と体重（`weight`）から肥満度を表す **`bmi`** 列を作る
- クラス（`class`：A〜Hの文字）は、そのままでは計算式に入れられないため、A＝8 〜 H＝1 の数値特徴量 **`class_num`** に変換する


In [ ]:
data["bmi"] = data["weight"] / (data["height"] / 100) ** 2

class_map = {"A": 8, "B": 7, "C": 6, "D": 5, "E": 4, "F": 3, "G": 2, "H": 1}
data["class_num"] = data["class"].map(class_map)

data[["name", "class", "class_num", "height", "weight", "bmi"]].head()


## 2. 構造化データの特徴量選択とAIの予測精度

第18講と同じように、データを無作為に学習データ（80%：542人）とテストデータ（20%：135人）に分ける。
そして、数学の点数（`math`）を予測するAI（`LinearRegression`）に**どの特徴量を入れるか**で、テストデータの誤差（**MAE**）がどう変わるか比べてみよう。


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

train = data.sample(frac=0.8, random_state=42)
test = data.drop(train.index)

feature_sets = [
    ("① 身長・体重のみ", ["height", "weight"]),
    ("② 国語の点数のみ", ["japanese"]),
    ("③ 国語＋身長・体重", ["japanese", "height", "weight"]),
    ("④ 国語＋クラス数値", ["japanese", "class_num"]),
]

for label, cols in feature_sets:
    model = LinearRegression().fit(train[cols], train["math"])
    pred = model.predict(test[cols])
    mae = mean_absolute_error(test["math"], pred)
    print(f"{label:18s} -> テストデータ MAE: {mae:.2f} 点")


- 数学と無関係な「身長・体重」だけを入れても、テストMAEは **12.15点** でほとんど役に立たない。
- 「国語」に無関係な「身長・体重」を足しても **10.61点 → 10.59点** と誤差はほぼ変わらない。
- 一方、クラス（A〜H）を数値化した特徴量 `class_num` を加えると、テストMAEは **4.74点** と半分以下に激減する！


## 3. 画像データ（数字の行列）と輪郭フィルターの体験

コンピュータにとって画像は「ピクセルの明るさ（0〜255）を並べた行列」である。
8×8マスの白黒画像（中央の4×4マスだけ明るい白い四角 `200`、周りは黒 `0`）を作り、隣り合うマスとの引き算（**輪郭抽出フィルター**）で何が起きるか実験しよう。


In [ ]:
img = np.zeros((8, 8), dtype=int)
img[2:6, 2:6] = 200  # 中央の4×4マスを明るい四角（200）にする

# 左右の隣のピクセルとの差（縦の輪郭フィルター）
v_edge = np.abs(img[:, 1:] - img[:, :-1])

# 上下の隣のピクセルとの差（横の輪郭フィルター）
h_edge = np.abs(img[1:, :] - img[:-1, :])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
sns.heatmap(img, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("① 元の8×8画像（中央が白い四角）")

sns.heatmap(v_edge, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1])
axes[1].set_title("② 縦の輪郭フィルター（左右の差）")

sns.heatmap(h_edge, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2])
axes[2].set_title("③ 横の輪郭フィルター（上下の差）")
plt.tight_layout()
plt.show()


左右の差をとると**縦の境界線（輪郭）**だけが `200` として浮かび上がり、上下の差をとると**横の境界線（輪郭）**だけが `200` として浮かび上がる。
畳み込みニューラルネットワーク（**CNN**）は、このようなフィルターの数字自体を大量の画像データから自動で学習する仕組みである。


## 4. 音声データ（波の合成）とフーリエ変換の体験

周波数 **50Hz（強さ1.0）** のサイン波と、周波数 **80Hz（強さ0.5）** のサイン波を足し合わせたギザギザな波を作り、**フーリエ変換**（`np.fft.rfft`）で元の周波数がどう抽出されるか確認しよう。


In [ ]:
# 0秒から0.5秒まで、1秒あたり1000回の細かさで時間を記録
x = np.linspace(0, 0.5, 500, endpoint=False)
y = np.sin(50 * 2 * np.pi * x) + 0.5 * np.sin(80 * 2 * np.pi * x)

# フーリエ変換で周波数ごとの強さ（振幅スペクトル）を計算
freqs = np.fft.rfftfreq(len(x), d=1/1000)
spectrum = np.abs(np.fft.rfft(y)) / (len(x) / 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(x[:150], y[:150])
axes[0].set_title("時間領域：混ざり合ったギザギザの波")
axes[0].set_xlabel("時間（秒）")

axes[1].plot(freqs[:150], spectrum[:150], color="crimson")
axes[1].set_title("周波数領域：フーリエ変換で抽出した特徴量")
axes[1].set_xlabel("周波数（Hz）")
plt.tight_layout()
plt.show()


左のギザギザな波形を見ただけでは何Hzの音が混ざっているか分からないが、フーリエ変換した右のグラフを見ると、**50Hz（高さ1.0）** と **80Hz（高さ0.5）** の2本の山がピタリと取り出せていることが分かる！


## 実習①：Pandasによるデータ操作と特徴量の作成

次の4つを順番に実行する。

1. `mhigh.csv` を読み込み、列名を `name`, `katakana`, `sex`, `grade`, `japanese`, `math`, `total`, `height`, `weight`, `class` に揃える
2. 条件抽出（`&`）を使い、「**3年生（`grade == 3`）かつクラスが `"A"`（`class == "A"`）**」の生徒が何人いるか求める
3. 身長と体重から **`bmi`** 列を作り、さらに国語と数学の差 **`diff_jm`**（`math - japanese`）の列を新しく作る
4. クラス `A〜H` を `8〜1` の数値に変換した **`class_num`** 列を作り、`data[["name", "class", "class_num", "bmi", "diff_jm"]].head()` を表示する


## 実習②：特徴量の選び方とAIの予測精度（テストMAE）

実習①で作った `data` を使い、次の4つを実行する。

1. `sample(frac=0.8, random_state=42)` と `drop` を使い、データを学習データ `train`（542人）とテストデータ `test`（135人）に分ける
2. `LinearRegression()` に体格の特徴量 `["height", "weight", "bmi"]` だけを学習させて数学（`math`）を予測し、テストデータの **MAE** を求める
3. 特徴量を `["japanese", "class_num"]` に変えて学習させ、テストデータの **MAE** がどれだけ小さくなるか求める
4. もし特徴量に `["japanese", "total"]`（成績合計）を入れてしまうとテストデータの **MAE** が `0.0` になることを確認し、なぜ実際の予測で `total` を特徴量にしてはいけないのか1〜2文で書く


## 実習③：画像データ（数字の行列）と輪郭フィルターの体験

次の4つを実行する。

1. `np.zeros((8, 8), dtype=int)` で 8×8 の黒い画像 `img2` を作り、`img2[1:7, 3:5] = 200` で中央に**縦長の白い棒**を描く
2. `sns.heatmap(img2, annot=True, fmt="d", cmap="Blues")` で、数字の行列が「縦棒の画像」になっていることを可視化する
3. 左右の差 `v_edge2 = np.abs(img2[:, 1:] - img2[:, :-1])`（縦の輪郭）と、上下の差 `h_edge2 = np.abs(img2[1:, :] - img2[:-1, :])`（横の輪郭）を計算し、それぞれの合計値（`v_edge2.sum()` と `h_edge2.sum()`）を比べる
4. 縦長の棒の画像に対して、縦の輪郭フィルターと横の輪郭フィルターのどちらが強く反応したか確認する
